# MVP — Pipeline de Dados na Nuvem

## Análise da carreira cinematográfica de Cary Grant

**Aluno:** Jennifer Silva  
**Curso:** Pós-graduação em Análise de Dados — PUC-Rio

---

## Objetivo do projeto

Este projeto tem como objetivo construir um pipeline de dados em nuvem para analisar informações relacionadas à carreira cinematográfica do ator Cary Grant.

O pipeline será desenvolvido utilizando o Databricks e seguirá uma estrutura baseada na Arquitetura Medalhão, organizando os dados nas camadas Bronze, Silver e Gold.

A partir dos dados coletados sobre filmes, avaliações, diretores e participação de Cary Grant nas produções cinematográficas, serão realizadas transformações e análises para compreender diferentes aspectos de sua carreira.

---

## Perguntas de negócio

1. Quais foram os filmes de Cary Grant com as melhores avaliações?

2. Como as avaliações dos filmes de Cary Grant se distribuem ao longo dos anos?

3. Quais gêneros cinematográficos aparecem com maior frequência em sua filmografia?

4. Quais foram as parcerias mais frequentes entre Cary Grant e diretores?

5. Quais diretores que trabalharam com Cary Grant tiveram os filmes com as melhores avaliações?

In [0]:
# Importação das bibliotecas necessárias

from pyspark.sql import functions as F
from pyspark.sql.types import *

In [0]:
# ============================================================
# CONFIGURAÇÃO DO PIPELINE
# ============================================================

# Volume onde os dados brutos serão armazenados
RAW_PATH = "/Volumes/workspace/default/cary_grant_raw"

# Identificador IMDb do Cary Grant
CARY_GRANT_ID = "nm0000026"

print("Volume configurado:", RAW_PATH)
print("Cary Grant:", CARY_GRANT_ID)

Volume configurado: /Volumes/workspace/default/cary_grant_raw
Cary Grant: nm0000026


In [0]:
# ============================================================
# CARGA DOS DADOS BRUTOS - CAMADA BRONZE
# ============================================================

# Caminho completo do arquivo CSV
FILE_PATH = f"{RAW_PATH}/cary_grant_filmes.csv"

# Leitura do arquivo CSV
df_bronze = (
    spark.read
    .format("csv")
    .option("header", True)
    .option("inferSchema", True)
    .option("sep", ",")
    .load(FILE_PATH)
)

# Visualização dos dados
display(df_bronze)

imdb_id,titulo,ano,generos,avaliacao_imdb,diretor
tt0053125,North by Northwest,1959,"Adventure, Mystery, Thriller",8.3,Alfred Hitchcock
tt0038787,Notorious,1946,"Drama, Film-Noir, Romance, Thriller",7.9,Alfred Hitchcock
tt0032904,The Philadelphia Story,1940,"Comedy, Romance",7.8,George Cukor
tt0029952,Bringing Up Baby,1938,"Comedy, Romance",7.8,Howard Hawks
tt0032599,His Girl Friday,1940,"Comedy, Drama, Romance",7.8,Howard Hawks
tt0056923,Charade,1963,"Comedy, Mystery, Romance, Thriller",7.8,Stanley Donen
tt0036613,Arsenic and Old Lace,1944,"Comedy, Crime, Romance",7.9,Frank Capra
tt0031931,Only Angels Have Wings,1939,"Adventure, Drama, Romance",7.6,Howard Hawks
tt0028597,The Awful Truth,1937,"Comedy, Romance",7.7,Leo McCarey
tt0031397,Gunga Din,1939,"Adventure, Comedy, War",7.8,George Stevens


In [0]:
# ============================================================
# VERIFICAÇÃO DO ESQUEMA DOS DADOS BRUTOS
# ============================================================

df_bronze.printSchema()

root
 |-- imdb_id: string (nullable = true)
 |-- titulo: string (nullable = true)
 |-- ano: integer (nullable = true)
 |-- generos: string (nullable = true)
 |-- avaliacao_imdb: double (nullable = true)
 |-- diretor: string (nullable = true)



In [0]:
# ============================================================
# TRANSFORMAÇÃO DOS DADOS - CAMADA SILVER
# ============================================================

df_silver = (
    df_bronze
    .withColumn("imdb_id", F.trim(F.col("imdb_id")))
    .withColumn("titulo", F.trim(F.col("titulo")))
    .withColumn("ano", F.col("ano").cast("int"))
    .withColumn("generos", F.trim(F.col("generos")))
    .withColumn("avaliacao_imdb", F.col("avaliacao_imdb").cast("double"))
    .withColumn("diretor", F.trim(F.col("diretor")))
    .dropDuplicates(["imdb_id"])
)

# Visualização dos dados tratados
display(df_silver)

imdb_id,titulo,ano,generos,avaliacao_imdb,diretor
tt0031397,Gunga Din,1939,"Adventure, Comedy, War",7.8,George Stevens
tt0044903,Monkey Business,1952,"Comedy, Sci-Fi, Romance",6.9,Howard Hawks
tt0036613,Arsenic and Old Lace,1944,"Comedy, Crime, Romance",7.9,Frank Capra
tt0032904,The Philadelphia Story,1940,"Comedy, Romance",7.8,George Cukor
tt0043949,Room for One More,1952,"Comedy, Drama, Romance",7.2,Norman Taurog
tt0038787,Notorious,1946,"Drama, Film-Noir, Romance, Thriller",7.9,Alfred Hitchcock
tt0039192,The Bishop's Wife,1947,"Comedy, Drama, Fantasy, Romance",7.6,Henry Koster
tt0053125,North by Northwest,1959,"Adventure, Mystery, Thriller",8.3,Alfred Hitchcock
tt0056923,Charade,1963,"Comedy, Mystery, Romance, Thriller",7.8,Stanley Donen
tt0031931,Only Angels Have Wings,1939,"Adventure, Drama, Romance",7.6,Howard Hawks


In [0]:
# ============================================================
# VERIFICAÇÃO DA QUALIDADE DOS DADOS - CAMADA SILVER
# ============================================================

quality = df_silver.agg(
    F.count("*").alias("total_registros"),
    F.countDistinct("imdb_id").alias("ids_distintos"),
    F.sum(F.col("imdb_id").isNull().cast("int")).alias("ids_nulos"),
    F.sum(F.col("titulo").isNull().cast("int")).alias("titulos_nulos"),
    F.sum(F.col("ano").isNull().cast("int")).alias("anos_nulos"),
    F.sum(F.col("generos").isNull().cast("int")).alias("generos_nulos"),
    F.sum(F.col("avaliacao_imdb").isNull().cast("int")).alias("avaliacoes_nulas"),
    F.sum(F.col("diretor").isNull().cast("int")).alias("diretores_nulos")
)

display(quality)

total_registros,ids_distintos,ids_nulos,titulos_nulos,anos_nulos,generos_nulos,avaliacoes_nulas,diretores_nulos
20,20,0,0,0,0,0,0


In [0]:
# ============================================================
# VERIFICAÇÃO COMPLEMENTAR DA QUALIDADE
# ============================================================

quality_complementar = df_silver.agg(
    F.sum(F.col("generos").isNull().cast("int")).alias("generos_nulos"),
    F.sum(F.col("avaliacao_imdb").isNull().cast("int")).alias("avaliacoes_nulas"),
    F.sum(F.col("diretor").isNull().cast("int")).alias("diretores_nulos"),
    F.min("avaliacao_imdb").alias("menor_avaliacao"),
    F.max("avaliacao_imdb").alias("maior_avaliacao")
)

display(quality_complementar)

generos_nulos,avaliacoes_nulas,diretores_nulos,menor_avaliacao,maior_avaliacao
0,0,0,5.9,8.3


In [0]:
# ============================================================
# PERSISTÊNCIA DA CAMADA SILVER - DELTA LAKE
# ============================================================

SILVER_TABLE = "workspace.default.cary_grant_silver"

df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(SILVER_TABLE)

print("Tabela Silver criada com sucesso:", SILVER_TABLE)

Tabela Silver criada com sucesso: workspace.default.cary_grant_silver


In [0]:
# ============================================================
# CONSULTA DA TABELA SILVER
# ============================================================

df_silver_catalogo = spark.table("workspace.default.cary_grant_silver")

display(df_silver_catalogo)

imdb_id,titulo,ano,generos,avaliacao_imdb,diretor
tt0031397,Gunga Din,1939,"Adventure, Comedy, War",7.8,George Stevens
tt0044903,Monkey Business,1952,"Comedy, Sci-Fi, Romance",6.9,Howard Hawks
tt0036613,Arsenic and Old Lace,1944,"Comedy, Crime, Romance",7.9,Frank Capra
tt0032904,The Philadelphia Story,1940,"Comedy, Romance",7.8,George Cukor
tt0043949,Room for One More,1952,"Comedy, Drama, Romance",7.2,Norman Taurog
tt0038787,Notorious,1946,"Drama, Film-Noir, Romance, Thriller",7.9,Alfred Hitchcock
tt0039192,The Bishop's Wife,1947,"Comedy, Drama, Fantasy, Romance",7.6,Henry Koster
tt0053125,North by Northwest,1959,"Adventure, Mystery, Thriller",8.3,Alfred Hitchcock
tt0056923,Charade,1963,"Comedy, Mystery, Romance, Thriller",7.8,Stanley Donen
tt0031931,Only Angels Have Wings,1939,"Adventure, Drama, Romance",7.6,Howard Hawks


In [0]:
# ============================================================
# CAMADA GOLD - FILMES COM MELHORES AVALIAÇÕES
# ============================================================

gold_avaliacoes = (
    df_silver_catalogo
    .select(
        "imdb_id",
        "titulo",
        "ano",
        "generos",
        "avaliacao_imdb",
        "diretor"
    )
    .orderBy(F.col("avaliacao_imdb").desc())
)

display(gold_avaliacoes)

imdb_id,titulo,ano,generos,avaliacao_imdb,diretor
tt0053125,North by Northwest,1959,"Adventure, Mystery, Thriller",8.3,Alfred Hitchcock
tt0036613,Arsenic and Old Lace,1944,"Comedy, Crime, Romance",7.9,Frank Capra
tt0038787,Notorious,1946,"Drama, Film-Noir, Romance, Thriller",7.9,Alfred Hitchcock
tt0032599,His Girl Friday,1940,"Comedy, Drama, Romance",7.8,Howard Hawks
tt0056923,Charade,1963,"Comedy, Mystery, Romance, Thriller",7.8,Stanley Donen
tt0032904,The Philadelphia Story,1940,"Comedy, Romance",7.8,George Cukor
tt0031397,Gunga Din,1939,"Adventure, Comedy, War",7.8,George Stevens
tt0029952,Bringing Up Baby,1938,"Comedy, Romance",7.8,Howard Hawks
tt0028597,The Awful Truth,1937,"Comedy, Romance",7.7,Leo McCarey
tt0039192,The Bishop's Wife,1947,"Comedy, Drama, Fantasy, Romance",7.6,Henry Koster


In [0]:
# ============================================================
# PERSISTÊNCIA DA GOLD - AVALIAÇÕES
# ============================================================

GOLD_AVALIACOES_TABLE = "workspace.default.cary_grant_gold_avaliacoes"

gold_avaliacoes.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(GOLD_AVALIACOES_TABLE)

print("Tabela Gold criada com sucesso:", GOLD_AVALIACOES_TABLE)

Tabela Gold criada com sucesso: workspace.default.cary_grant_gold_avaliacoes


In [0]:
# ============================================================
# CAMADA GOLD - FREQUÊNCIA DOS GÊNEROS
# ============================================================

gold_generos = (
    df_silver_catalogo
    .select(
        "imdb_id",
        "titulo",
        "avaliacao_imdb",
        F.explode(
            F.split(F.col("generos"), ", ")
        ).alias("genero")
    )
)

# Contagem de filmes por gênero
gold_generos_contagem = (
    gold_generos
    .groupBy("genero")
    .agg(
        F.countDistinct("imdb_id").alias("quantidade_filmes"),
        F.round(F.avg("avaliacao_imdb"), 2).alias("avaliacao_media")
    )
    .orderBy(F.col("quantidade_filmes").desc())
)

display(gold_generos_contagem)

genero,quantidade_filmes,avaliacao_media
Romance,18,7.36
Comedy,14,7.33
Drama,6,7.58
Thriller,5,7.74
Mystery,4,7.7
Adventure,3,7.9
Film-Noir,2,7.6
War,2,7.5
Fantasy,1,7.6
Crime,1,7.9


In [0]:
# ============================================================
# PERSISTÊNCIA DA GOLD - GÊNEROS
# ============================================================

GOLD_GENEROS_TABLE = "workspace.default.cary_grant_gold_generos"

gold_generos_contagem.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(GOLD_GENEROS_TABLE)

print("Tabela Gold de gêneros criada com sucesso:", GOLD_GENEROS_TABLE)

Tabela Gold de gêneros criada com sucesso: workspace.default.cary_grant_gold_generos


In [0]:
# ============================================================
# CAMADA GOLD - ANÁLISE DOS DIRETORES
# ============================================================

gold_diretores = (
    df_silver_catalogo
    .groupBy("diretor")
    .agg(
        F.countDistinct("imdb_id").alias("quantidade_filmes"),
        F.round(F.avg("avaliacao_imdb"), 2).alias("avaliacao_media"),
        F.round(F.max("avaliacao_imdb"), 2).alias("melhor_avaliacao")
    )
    .orderBy(F.col("quantidade_filmes").desc())
)

display(gold_diretores)

diretor,quantidade_filmes,avaliacao_media,melhor_avaliacao
Howard Hawks,4,7.53,7.8
Alfred Hitchcock,4,7.73,8.3
Leo McCarey,2,7.55,7.7
Blake Edwards,1,7.2,7.2
Charles Walters,1,6.6,6.6
George Stevens,1,7.8,7.8
Stanley Donen,1,7.8,7.8
George Cukor,1,7.8,7.8
Frank Capra,1,7.9,7.9
Henry Koster,1,7.6,7.6


In [0]:
# ============================================================
# PERSISTÊNCIA DA GOLD - DIRETORES
# ============================================================

GOLD_DIRETORES_TABLE = "workspace.default.cary_grant_gold_diretores"

gold_diretores.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(GOLD_DIRETORES_TABLE)

print("Tabela Gold de diretores criada com sucesso:", GOLD_DIRETORES_TABLE)

Tabela Gold de diretores criada com sucesso: workspace.default.cary_grant_gold_diretores


In [0]:
# ============================================================
# ANÁLISE 1 - FILMES COM MELHORES AVALIAÇÕES
# ============================================================

analise_1 = (
    spark.table("workspace.default.cary_grant_gold_avaliacoes")
    .orderBy(F.col("avaliacao_imdb").desc())
    .limit(10)
)

display(analise_1)

imdb_id,titulo,ano,generos,avaliacao_imdb,diretor
tt0053125,North by Northwest,1959,"Adventure, Mystery, Thriller",8.3,Alfred Hitchcock
tt0036613,Arsenic and Old Lace,1944,"Comedy, Crime, Romance",7.9,Frank Capra
tt0038787,Notorious,1946,"Drama, Film-Noir, Romance, Thriller",7.9,Alfred Hitchcock
tt0056923,Charade,1963,"Comedy, Mystery, Romance, Thriller",7.8,Stanley Donen
tt0029952,Bringing Up Baby,1938,"Comedy, Romance",7.8,Howard Hawks
tt0032904,The Philadelphia Story,1940,"Comedy, Romance",7.8,George Cukor
tt0031397,Gunga Din,1939,"Adventure, Comedy, War",7.8,George Stevens
tt0032599,His Girl Friday,1940,"Comedy, Drama, Romance",7.8,Howard Hawks
tt0028597,The Awful Truth,1937,"Comedy, Romance",7.7,Leo McCarey
tt0039192,The Bishop's Wife,1947,"Comedy, Drama, Fantasy, Romance",7.6,Henry Koster


In [0]:
# ============================================================
# ANÁLISE 2 - AVALIAÇÕES AO LONGO DOS ANOS
# ============================================================

analise_2 = (
    spark.table("workspace.default.cary_grant_silver")
    .groupBy("ano")
    .agg(
        F.count("imdb_id").alias("quantidade_filmes"),
        F.round(F.avg("avaliacao_imdb"), 2).alias("avaliacao_media")
    )
    .orderBy("ano")
)

display(analise_2)

ano,quantidade_filmes,avaliacao_media
1937,1,7.7
1938,1,7.8
1939,2,7.7
1940,2,7.8
1941,1,7.3
1944,1,7.9
1946,1,7.9
1947,1,7.6
1952,2,7.05
1953,1,5.9


In [0]:
# ============================================================
# ANÁLISE 3 - FREQUÊNCIA DOS GÊNEROS
# ============================================================

analise_3 = (
    spark.table("workspace.default.cary_grant_gold_generos")
    .orderBy(
        F.col("quantidade_filmes").desc(),
        F.col("avaliacao_media").desc()
    )
)

display(analise_3)

genero,quantidade_filmes,avaliacao_media
Romance,18,7.36
Comedy,14,7.33
Drama,6,7.58
Thriller,5,7.74
Mystery,4,7.7
Adventure,3,7.9
Film-Noir,2,7.6
War,2,7.5
Crime,1,7.9
Fantasy,1,7.6


In [0]:
# ============================================================
# ANÁLISE 4 - DIRETORES COM QUEM CARY GRANT MAIS TRABALHOU
# ============================================================

analise_4 = (
    spark.table("workspace.default.cary_grant_gold_diretores")
    .orderBy(
        F.col("quantidade_filmes").desc(),
        F.col("avaliacao_media").desc()
    )
)

display(analise_4)

diretor,quantidade_filmes,avaliacao_media,melhor_avaliacao
Alfred Hitchcock,4,7.73,8.3
Howard Hawks,4,7.52,7.8
Leo McCarey,2,7.55,7.7
Frank Capra,1,7.9,7.9
George Stevens,1,7.8,7.8
Stanley Donen,1,7.8,7.8
George Cukor,1,7.8,7.8
Henry Koster,1,7.6,7.6
Norman Taurog,1,7.2,7.2
Blake Edwards,1,7.2,7.2


In [0]:
# ============================================================
# ANÁLISE 5 - DIRETORES COM OS FILMES MAIS BEM AVALIADOS
# ============================================================

analise_5 = (
    spark.table("workspace.default.cary_grant_gold_diretores")
    .select(
        "diretor",
        "quantidade_filmes",
        "avaliacao_media",
        "melhor_avaliacao"
    )
    .orderBy(
        F.col("melhor_avaliacao").desc(),
        F.col("avaliacao_media").desc()
    )
)

display(analise_5)

diretor,quantidade_filmes,avaliacao_media,melhor_avaliacao
Alfred Hitchcock,4,7.73,8.3
Frank Capra,1,7.9,7.9
George Stevens,1,7.8,7.8
Stanley Donen,1,7.8,7.8
George Cukor,1,7.8,7.8
Howard Hawks,4,7.52,7.8
Leo McCarey,2,7.55,7.7
Henry Koster,1,7.6,7.6
Norman Taurog,1,7.2,7.2
Blake Edwards,1,7.2,7.2
